# Inspect Rejected Records

## Spark setup cell

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("InspectRejectedRecords")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.session.timeZone", "UTC")

## Inspect individual rejected records

In [ ]:
REJECTED_RECORDS_PATH = (
    "s3a://delta-data/metadata/rejected_records"
)

rejected_records = (
    spark.read
    .format("delta")
    .load(REJECTED_RECORDS_PATH)
)

rejected_records.printSchema()

(
    rejected_records
    .orderBy(F.desc("rejected_at"))
    .show(100, truncate=False)
)

## Summarize failures by reason

In [ ]:
(
    rejected_records
    .select(
        "dataset",
        "schema_version",
        F.explode("rejection_reasons").alias(
            "rejection_reason"
        ),
    )
    .groupBy(
        "dataset",
        "schema_version",
        "rejection_reason",
    )
    .count()
    .orderBy(
        F.desc("count"),
        "dataset",
        "rejection_reason",
    )
    .show(truncate=False)
)

## Stop Spark

In [ ]:
spark.stop()